# Makam-Conditioned Music Generation with Markov Chains

## Overview

This notebook presents a reproducible symbolic music generation pipeline for Turkish makam music using MusicXML representations and Markov Chain-based probabilistic models.

The input dataset consists of **3,000 MusicXML documents derived from the SymbTr v3.0 Turkish makam music corpus**. Each MusicXML file represents a single musical composition containing symbolic musical information, including pitch, duration, rhythmic structure, and contextual metadata.

The workflow extracts symbolic musical events from MusicXML files, organizes compositions according to makam characteristics, and trains makam-conditioned Markov Chain models to learn statistical relationships between musical events.

Two different makam-conditioned Markov modeling strategies are investigated:

### Data Availability-Based Markov Models

The first strategy selects makam categories according to their representation frequency within the corpus.

These models focus on learning melodic transition patterns from makam classes with larger numbers of available compositions.

### Musical Diversity-Based Markov Models

The second strategy selects makam categories to represent a broader range of Turkish makam characteristics.

These models investigate how musical diversity influences generated melodic structures.

The generated symbolic compositions are analyzed according to:

- Pitch transition patterns
- Microtonal pitch alterations
- Octave movements
- Rhythmic duration patterns
- Rest structures
- Local melodic transition behaviors

The complete generation pipeline follows:



## Importing the Required Libraries

This notebook uses Python libraries for MusicXML parsing, symbolic music representation, probabilistic modeling, data organization, audio generation, and result visualization.

The imported libraries provide the following functionalities:

- **File and path management:** Handling corpus directories, input files, and generated outputs.
- **XML processing:** Parsing MusicXML documents and extracting symbolic musical events.
- **Symbolic representation:** Creating structured `MusicEvent` objects and converting musical events into token sequences.
- **Data organization:** Managing makam-based musical sequences for statistical analysis and model training.
- **Probabilistic modeling:** Implementing makam-conditioned Markov Chain models using Python data structures.
- **Random sampling:** Generating new symbolic musical sequences based on learned transition probabilities.
- **Audio processing:** Synthesizing generated symbolic sequences into audio representations.
- **Visualization:** Exploring corpus statistics and generated model outputs.

Two Markov Chain modeling strategies are investigated:

- **Data Availability-Based Models:** Training models using makam categories selected according to corpus representation frequency.
- **Musical Diversity-Based Models:** Training models using makam categories selected to represent broader musical characteristics.

The Markov Chain implementation is developed using transparent Python-based approaches without requiring specialized end-to-end music generation frameworks. This provides a reproducible and interpretable methodology for symbolic Turkish makam music generation.

In [117]:
from pathlib import Path

from collections import Counter, defaultdict

from dataclasses import dataclass, asdict

from typing import Optional, List, Dict


import json

import math

import random

import re

import unicodedata


import xml.etree.ElementTree as ET


import numpy as np

import pandas as pd

import matplotlib.pyplot as plt


from tqdm import tqdm


from scipy.io.wavfile import write

## Project Configuration and Reproducibility Setup

This section defines the project configuration, directory structure, and output locations used throughout the notebook.

All paths are constructed relative to the main project directory to ensure reproducibility across different execution environments.

The configuration manages:

- Original MusicXML corpus locations,
- Processed symbolic sequence storage,
- Generated MusicXML outputs,
- Generated MIDI files,
- Generated WAV audio files,
- AI-generated makam composition outputs.

Centralizing all project paths improves maintainability and prevents dependency on environment-specific absolute paths.

This configuration allows the complete symbolic music analysis and AI generation workflow to be reproduced after restarting the notebook environment.

## Configuring the MusicXML Dataset Directory

This notebook uses the MusicXML corpus generated during the previous SymbTr TXT-to-MusicXML conversion stage.

The input dataset consists of **3,000 MusicXML files**, where each document represents a single Turkish makam music composition.

The project directory structure separates:

- Original MusicXML corpus,
- Extracted symbolic representations,
- Trained Markov Chain models,
- Generated MusicXML compositions,
- Generated MIDI files,
- Generated WAV audio files.

This organization provides a reproducible workflow by keeping input data, intermediate processing results, trained models, and generated outputs in separate directories.

The expected workflow is:



In [118]:
from pathlib import Path


# =====================================================
# Project Root Directory
# =====================================================

project_directory = Path.cwd().parent


# =====================================================
# Dataset Directory
# =====================================================

musicxml_directory = (
    project_directory
    / "data"
    / "musicxml"
    / "xml_files"
)


# =====================================================
# Validate Dataset
# =====================================================

if not musicxml_directory.exists():
    raise FileNotFoundError(
        "MusicXML corpus directory was not found."
    )


# =====================================================
# Output Directories
# =====================================================

processed_data_directory = (
    project_directory
    / "data"
    / "processed"
)

models_directory = (
    project_directory
    / "models"
)

generated_music_directory = (
    project_directory
    / "data"
    / "generated_music"
)

generated_xml_directory = (
    generated_music_directory
    / "musicxml"
)

generated_midi_directory = (
    generated_music_directory
    / "midi"
)

generated_wav_directory = (
    generated_music_directory
    / "wav"
)


for directory in [
    processed_data_directory,
    models_directory,
    generated_xml_directory,
    generated_midi_directory,
    generated_wav_directory,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# =====================================================
# Generated Files
# =====================================================

generated_xml_file = (
    generated_xml_directory
    / "AI_generated_makam.xml"
)

generated_midi_file = (
    generated_midi_directory
    / "AI_generated_makam.mid"
)

generated_wav_file = (
    generated_wav_directory
    / "AI_generated_makam.wav"
)


# =====================================================
# Configuration Summary
# =====================================================

print("=" * 70)
print("Project Configuration Summary")
print("=" * 70)
print("✓ Project directories configured")
print("✓ MusicXML corpus directory loaded")
print("✓ Output directories prepared")
print("✓ Markov model directory prepared")
print("✓ Generated music output enabled")
print("=" * 70)

Project Configuration Summary
✓ Project directories configured
✓ MusicXML corpus directory loaded
✓ Output directories prepared
✓ Markov model directory prepared
✓ Generated music output enabled


In [119]:
from pathlib import Path


# =====================================================
# Project Root Directory
# =====================================================

project_directory = Path(
    r"D:\TDC-Analysis-Book"
)



# =====================================================
# Dataset Directory
# =====================================================

musicxml_directory = (
    project_directory
    /
    "data"
    /
    "musicxml"
    /
    "xml_files"
)



# =====================================================
# Validate Dataset
# =====================================================

if not musicxml_directory.exists():

    raise FileNotFoundError(
        "MusicXML corpus directory was not found."
    )



# =====================================================
# Output Directories
# =====================================================

processed_data_directory = (
    project_directory
    /
    "data"
    /
    "processed"
)


models_directory = (
    project_directory
    /
    "models"
)


generated_music_directory = (
    project_directory
    /
    "data"
    /
    "generated_music"
)


generated_xml_directory = (
    generated_music_directory
    /
    "musicxml"
)


generated_midi_directory = (
    generated_music_directory
    /
    "midi"
)


generated_wav_directory = (
    generated_music_directory
    /
    "wav"
)



for directory in [
    processed_data_directory,
    models_directory,
    generated_xml_directory,
    generated_midi_directory,
    generated_wav_directory
]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )

# =====================================================
# Generated Files
# =====================================================

generated_xml_file = (
    generated_xml_directory
    /
    "AI_generated_makam.xml"
)


generated_midi_file = (
    generated_midi_directory
    /
    "AI_generated_makam.mid"
)


generated_wav_file = (
    generated_wav_directory
    /
    "AI_generated_makam.wav"
)



# =====================================================
# Configuration Summary
# =====================================================

print("=" * 70)

print(
    "Project Configuration Summary"
)

print("=" * 70)


print(
    "✓ Project directories configured"
)

print(
    "✓ MusicXML corpus directory loaded"
)

print(
    "✓ Processed data directory prepared"
)

print(
    "✓ Markov model directory prepared"
)

print(
    "✓ Generated music output enabled"
)

print("=" * 70)

Project Configuration Summary
✓ Project directories configured
✓ MusicXML corpus directory loaded
✓ Processed data directory prepared
✓ Markov model directory prepared
✓ Generated music output enabled


In [120]:
xml_check = list(
    project_directory.rglob("*.xml")
)

print(f"XML files found in project: {len(xml_check):,}")

for file in xml_check[:5]:
    print(file.relative_to(project_directory))

XML files found in project: 9,281
.venv\Lib\site-packages\music21\corpus\bach\bwv67.4.xml
.venv\Lib\site-packages\music21\corpus\bach\bwv69.6.xml
.venv\Lib\site-packages\music21\corpus\ciconia\quod_jactatur.xml
.venv\Lib\site-packages\music21\corpus\corelli\opus3no1\1grave.xml
.venv\Lib\site-packages\music21\corpus\demos\drum_sample.xml


## Discovering the Generated MusicXML Corpus

This section locates the generated MusicXML corpus and verifies that the symbolic dataset is available for analysis.

The search is restricted to the designated MusicXML dataset directory, ensuring that only corpus files are included. Both `.xml` and `.musicxml` formats are supported.

The discovered documents constitute the input corpus for the subsequent stages of the workflow, including symbolic event extraction, sequence construction, Markov chain training, and symbolic music generation.

**Output.** This cell reports the number of MusicXML files detected and confirms that the corpus is ready for processing.

In [121]:
from pathlib import Path

xml_files = sorted(
    musicxml_directory.rglob("*.xml")
)

total_size_bytes = sum(
    xml_file.stat().st_size
    for xml_file in xml_files
)

total_size_mb = total_size_bytes / (1024 ** 2)

print("=" * 60)
print("Generated MusicXML Corpus Discovery")
print("=" * 60)
print(f"Total MusicXML files : {len(xml_files):,}")
print(f"Corpus size          : {total_size_mb:.2f} MB")
print("Status               : Corpus successfully located")
print("=" * 60)

Generated MusicXML Corpus Discovery
Total MusicXML files : 3,000
Corpus size          : 131.74 MB
Status               : Corpus successfully located


## Defining the Symbolic Music Event Representation

The generated MusicXML corpus contains structured symbolic representations of Turkish makam music compositions. Before constructing musical sequences for Markov Chain modeling, each MusicXML note or rest is transformed into a consistent symbolic event representation.

Each `MusicEvent` object corresponds to a single `<note>` element extracted from a MusicXML document. Since a MusicXML `<note>` element may represent either a pitched note or a rest, pitch-related attributes are defined as optional. Pitched events contain `<step>`, an optional `<alter>`, and `<octave>` information, whereas rest events are represented through the `<rest>` element without requiring pitch information.

The symbolic event representation contains the following attributes:

- **Pitch step:** The diatonic note name extracted from the MusicXML `<step>` element.
- **Microtonal alteration:** The semitone-based pitch deviation extracted from the optional `<alter>` element.
- **Octave:** Register information extracted from the `<octave>` element.
- **Normalized duration:** Rhythmic duration calculated from the MusicXML `<duration>` and `<divisions>` values.
- **Rest status:** Indicates whether the event represents a pitched note or a musical rest.
- **Lyric annotation:** Optional textual information extracted from the MusicXML `<lyric>` element.
- **Measure number:** The measure in which the musical event occurs.
- **Composition metadata:** Makam, usul, musical form, composer, and source-file information associated with the composition.

Rhythmic durations are represented using exact fractional values. This prevents floating-point rounding errors and preserves rhythmic relationships when the MusicXML corpus is transformed into Markov Chain sequences.

Two derived token representations are also provided:

- `pitch_token` creates a readable representation of the pitch or rest.
- `event_token` combines pitch, microtonal alteration, octave, duration, and rest status into a stable token suitable for Markov Chain training.

This event structure provides a consistent intermediate representation between the MusicXML parser and the Markov Chain model.

In [122]:
from dataclasses import dataclass
from fractions import Fraction
from typing import Optional


@dataclass(frozen=True)
class MusicEvent:
    """
    Structured symbolic representation of a MusicXML
    note or rest event.
    """

    # Event type
    is_rest: bool

    # Exact duration measured in quarter-note units
    duration: Fraction

    # Optional pitch information
    step: Optional[str] = None
    alter: Optional[float] = None
    octave: Optional[int] = None

    # Optional textual information
    lyric: Optional[str] = None

    # Structural information
    measure_number: Optional[int] = None

    # Composition-level metadata
    makam: Optional[str] = None
    usul: Optional[str] = None
    form: Optional[str] = None
    composer: Optional[str] = None
    source_file: Optional[str] = None

    def __post_init__(self):
        """
        Validate the internal consistency of the event.
        """

        if self.duration <= 0:
            raise ValueError(
                "A musical event must have a positive duration."
            )

        if self.is_rest:
            return

        if self.step is None:
            raise ValueError(
                "A pitched musical event must contain a pitch step."
            )

        if self.octave is None:
            raise ValueError(
                "A pitched musical event must contain an octave value."
            )

        valid_steps = {
            "A",
            "B",
            "C",
            "D",
            "E",
            "F",
            "G",
        }

        normalized_step = self.step.upper()

        if normalized_step not in valid_steps:
            raise ValueError(
                f"Invalid MusicXML pitch step: {self.step}"
            )

        object.__setattr__(
            self,
            "step",
            normalized_step,
        )

        if self.alter is None:
            object.__setattr__(
                self,
                "alter",
                0.0,
            )

    @property
    def pitch_token(self) -> str:
        """
        Return a readable pitch representation.
        """

        if self.is_rest:
            return "REST"

        return (
            f"{self.step}"
            f"{self.alter:+.8f}"
            f"_{self.octave}"
        )

    @property
    def duration_token(self) -> str:
        """
        Return the exact rhythmic duration as a fraction.
        """

        return (
            f"{self.duration.numerator}/"
            f"{self.duration.denominator}"
        )

    @property
    def event_token(self) -> str:
        """
        Convert the event into a stable Markov Chain token.
        """

        if self.is_rest:
            return (
                f"REST|"
                f"{self.duration_token}"
            )

        return (
            f"{self.step}|"
            f"{self.alter:.8f}|"
            f"{self.octave}|"
            f"{self.duration_token}"
        )

## Extracting Composition Metadata from MusicXML Filenames

Each generated MusicXML document preserves the original SymbTr filename convention, allowing composition-level metadata to be recovered directly from the filename.

Although the MusicXML documents also contain embedded metadata, the filename provides a lightweight and consistent mechanism for identifying each composition before parsing the XML content.

The filename is expected to follow the SymbTr naming convention:

```
makam--form--usul--title--composer.xml
```

From this naming structure, the notebook extracts the following attributes:

- **Makam**
- **Musical form**
- **Usul (rhythmic cycle)**
- **Composition title**
- **Composer**

The extracted metadata is associated with every symbolic musical event generated from the corresponding MusicXML document. These attributes can subsequently be used for corpus organization, statistical analysis, filtering, and makam-conditioned Markov Chain music generation.

If a filename does not fully conform to the expected naming convention, unavailable fields are assigned `None`, allowing the remaining workflow to continue without interruption.

In [123]:
from pathlib import Path


def extract_metadata_from_filename(filename):
    """
    Extract composition metadata from a SymbTr MusicXML filename.

    Expected filename format:
    makam--form--usul--title--composer.xml
    """

    stem = Path(filename).stem
    parts = stem.split("--")

    keys = (
        "makam",
        "form",
        "usul",
        "title",
        "composer",
    )

    metadata = dict.fromkeys(keys)

    for key, value in zip(keys, parts):
        metadata[key] = value.strip() or None

    return metadata

### Metadata Extraction Validation

To verify the correctness of the filename parser, a representative SymbTr MusicXML filename is processed using the metadata extraction function.

The extracted metadata should correctly identify the makam, musical form, usul, composition title, and composer encoded in the filename. This validation demonstrates that the parser correctly interprets the standard SymbTr naming convention before it is applied to the complete MusicXML corpus.

In [124]:
sample_filename = (
    "acem--ilahi--duyek--aldanma_dunya--zekai_dede.xml"
)

sample_metadata = extract_metadata_from_filename(
    sample_filename
)

print("=" * 60)
print("Filename Metadata Validation")
print("=" * 60)
print(f"Filename : {sample_filename}")
print()

for key, value in sample_metadata.items():
    print(f"{key.capitalize():10}: {value}")

print("=" * 60)

Filename Metadata Validation
Filename : acem--ilahi--duyek--aldanma_dunya--zekai_dede.xml

Makam     : acem
Form      : ilahi
Usul      : duyek
Title     : aldanma_dunya
Composer  : zekai_dede


## Inspecting the MusicXML Metadata Structure

Although the primary composition metadata is obtained from the SymbTr filename convention, the generated MusicXML documents also contain embedded descriptive information.

Before parsing the complete corpus, a representative MusicXML document is inspected to verify that the expected metadata elements have been correctly generated.

Typical metadata elements include:

- **`<work-title>`**, describing the composition title;
- **`<creator>`**, identifying contributors such as the composer;
- additional descriptive elements generated during the conversion process.

This inspection serves as a structural validation step to confirm that the generated MusicXML documents contain the expected metadata before symbolic event extraction begins.

The extracted values are displayed for verification only and are not used as the primary metadata source for the subsequent Markov Chain modeling workflow.

In [125]:
# Inspect the metadata structure of a representative MusicXML document

sample_xml = xml_files[0]

tree = ET.parse(sample_xml)
root = tree.getroot()

metadata_tags = (
    "work-title",
    "creator",
)

print("=" * 60)
print("MusicXML Metadata Inspection")
print("=" * 60)
print(f"Sample file : {sample_xml.name}")
print()

found = False

for element in root.iter():
    tag = element.tag.split("}")[-1]

    if tag in metadata_tags and element.text:
        print(f"{tag:<12}: {element.text}")
        found = True

if not found:
    print("No metadata elements were found.")

print("=" * 60)

MusicXML Metadata Inspection
Sample file : acem--ilahi--duyek--aldanma_dunya--zekai_dede.xml

work-title  : acem--ilahi--duyek--aldanma_dunya--zekai_dede


## Validating Symbolic Event Extraction

Before processing the complete MusicXML corpus, the symbolic event extraction workflow is validated using a representative MusicXML document.

During this validation, the notebook:

- extracts composition metadata from the SymbTr filename,
- parses the MusicXML document,
- converts notes and rests into `MusicEvent` objects,
- preserves pitch, microtonal alteration, rhythmic duration, and lyric information,
- and displays the extracted symbolic events to verify that the resulting representation is suitable for subsequent Markov Chain music generation.

The output reports the extracted metadata, the total number of symbolic events, and a sample of the generated `MusicEvent` objects for manual inspection.

In [126]:
import xml.etree.ElementTree as ET
from fractions import Fraction


def extract_music_events_from_xml(xml_file, metadata=None):
    """
    Parse a MusicXML document and convert its note and rest
    elements into MusicEvent objects.
    """

    metadata = metadata or {}

    tree = ET.parse(xml_file)
    root = tree.getroot()

    events = []

    divisions = 1

    for measure in root.iter():
        measure_tag = measure.tag.split("}")[-1]

        if measure_tag != "measure":
            continue

        measure_number_raw = measure.get("number")

        try:
            measure_number = int(measure_number_raw)
        except (TypeError, ValueError):
            measure_number = None

        for child in measure:
            child_tag = child.tag.split("}")[-1]

            if child_tag == "attributes":
                for element in child:
                    tag = element.tag.split("}")[-1]

                    if tag == "divisions" and element.text:
                        try:
                            divisions = int(element.text)
                        except ValueError:
                            divisions = 1

            if child_tag != "note":
                continue

            is_rest = False
            step = None
            alter = 0.0
            octave = None
            duration_value = None
            lyric = None

            for note_element in child:
                note_tag = note_element.tag.split("}")[-1]

                if note_tag == "rest":
                    is_rest = True

                elif note_tag == "pitch":
                    for pitch_element in note_element:
                        pitch_tag = pitch_element.tag.split("}")[-1]

                        if pitch_tag == "step":
                            step = pitch_element.text

                        elif pitch_tag == "alter":
                            try:
                                alter = float(pitch_element.text)
                            except (TypeError, ValueError):
                                alter = 0.0

                        elif pitch_tag == "octave":
                            try:
                                octave = int(pitch_element.text)
                            except (TypeError, ValueError):
                                octave = None

                elif note_tag == "duration":
                    try:
                        duration_value = int(note_element.text)
                    except (TypeError, ValueError):
                        duration_value = None

                elif note_tag == "lyric":
                    for lyric_element in note_element:
                        lyric_tag = lyric_element.tag.split("}")[-1]

                        if lyric_tag == "text":
                            lyric = lyric_element.text
                            break

            if duration_value is None or duration_value <= 0:
                continue

            normalized_duration = Fraction(
                duration_value,
                divisions,
            )

            event = MusicEvent(
                is_rest=is_rest,
                duration=normalized_duration,
                step=None if is_rest else step,
                alter=None if is_rest else alter,
                octave=None if is_rest else octave,
                lyric=lyric,
                measure_number=measure_number,
                makam=metadata.get("makam"),
                usul=metadata.get("usul"),
                form=metadata.get("form"),
                composer=metadata.get("composer"),
                source_file=xml_file.name,
            )

            events.append(event)

    return events

In [127]:
# Validate symbolic event extraction using a representative MusicXML document

sample_xml = xml_files[0]

print("=" * 60)
print("Symbolic Event Extraction Validation")
print("=" * 60)
print(f"Sample file : {sample_xml.name}")
print("=" * 60)

# Extract composition metadata
sample_metadata = extract_metadata_from_filename(
    sample_xml.name
)

print("Composition Metadata")
print("-" * 60)

for key, value in sample_metadata.items():
    print(f"{key.capitalize():10}: {value}")

print("-" * 60)

# Extract symbolic musical events
sample_events = extract_music_events_from_xml(
    sample_xml,
    metadata=sample_metadata,
)

print(f"Total symbolic events : {len(sample_events):,}")

print("\nFirst five symbolic events")
print("-" * 60)

for i, event in enumerate(sample_events[:5], start=1):
    print(f"{i:>2}. {event}")

print("=" * 60)

Symbolic Event Extraction Validation
Sample file : acem--ilahi--duyek--aldanma_dunya--zekai_dede.xml
Composition Metadata
------------------------------------------------------------
Makam     : acem
Form      : ilahi
Usul      : duyek
Title     : aldanma_dunya
Composer  : zekai_dede
------------------------------------------------------------
Total symbolic events : 262

First five symbolic events
------------------------------------------------------------
 1. MusicEvent(is_rest=False, duration=Fraction(714, 1), step='C', alter=1.59, octave=5, lyric='Al', measure_number=1, makam='acem', usul='duyek', form='ilahi', composer='zekai_dede', source_file='acem--ilahi--duyek--aldanma_dunya--zekai_dede.xml')
 2. MusicEvent(is_rest=False, duration=Fraction(536, 1), step='F', alter=1.7, octave=5, lyric='dan', measure_number=1, makam='acem', usul='duyek', form='ilahi', composer='zekai_dede', source_file='acem--ilahi--duyek--aldanma_dunya--zekai_dede.xml')
 3. MusicEvent(is_rest=False, duration=

## Inspecting the Generated MusicXML Structure

Before defining the complete symbolic event parser, a representative generated MusicXML document is inspected to verify its internal hierarchy.

This structural inspection identifies:

- the MusicXML root element,
- whether an XML namespace is present,
- the principal score-level elements,
- musical parts,
- measures,
- note and rest elements,
- pitch components,
- duration values,
- and lyric annotations.

Understanding the exact hierarchy of the generated documents ensures that the subsequent parser is designed according to the current MusicXML representation rather than assumptions derived from the original SymbTr TXT format.

The inspection displays a limited number of XML elements to keep the notebook output concise while confirming that the expected score structure is present.

In [128]:
# Inspect the structure of a representative generated MusicXML document

tree = ET.parse(sample_xml)
root = tree.getroot()

root_tag = root.tag.split("}")[-1]

namespace = (
    root.tag.split("}")[0].removeprefix("{")
    if root.tag.startswith("{")
    else None
)

elements = list(root.iter())

print("=" * 60)
print("Generated MusicXML Structure Inspection")
print("=" * 60)
print(f"Sample file    : {sample_xml.name}")
print(f"Root element   : {root_tag}")
print(
    f"XML namespace  : {namespace if namespace else 'Not present'}"
)
print(f"Total elements : {len(elements):,}")
print("=" * 60)

print("First 20 XML elements")
print("-" * 60)

for index, element in enumerate(
    elements[:20],
    start=1,
):
    tag = element.tag.split("}")[-1]
    print(f"{index:>2}. {tag}")

if len(elements) > 20:
    print(f"... ({len(elements) - 20:,} additional elements)")

print("=" * 60)

Generated MusicXML Structure Inspection
Sample file    : acem--ilahi--duyek--aldanma_dunya--zekai_dede.xml
Root element   : score-partwise
XML namespace  : Not present
Total elements : 1,732
First 20 XML elements
------------------------------------------------------------
 1. score-partwise
 2. work
 3. work-title
 4. part-list
 5. score-part
 6. part-name
 7. part
 8. measure
 9. note
10. pitch
11. step
12. alter
13. octave
14. duration
15. lyric
16. text
17. note
18. pitch
19. step
20. alter
... (1,712 additional elements)


### Converting Symbolic Events into Markov Tokens

The extracted `MusicEvent` objects are converted into symbolic tokens that serve as the input representation for first-order Markov Chain modeling.

Each token encodes the symbolic information required to represent a musical event, including pitch (or rest), microtonal alteration, octave, and rhythmic duration. This standardized representation enables consistent estimation of transition probabilities between consecutive musical states.

**Output.** This cell converts each sequence of `MusicEvent` objects into a sequence of symbolic tokens for subsequent Markov Chain training.

In [129]:
def events_to_token_sequence(events):
    """
    Convert MusicEvent objects into symbolic tokens.
    """

    return [
        event.event_token
        for event in events
    ]

## Extracting Symbolic Events from MusicXML Documents

The generated MusicXML documents are parsed to transform individual `<note>` elements into structured `MusicEvent` objects.

For each musical event, the parser extracts:

- pitch step,
- microtonal alteration,
- octave information,
- normalized rhythmic duration,
- rest status,
- lyric annotations,
- measure number,
- and composition-level metadata obtained from the SymbTr filename.

MusicXML duration values are interpreted together with the `<divisions>` element. The resulting durations are stored as exact fractional values measured in quarter-note units. This preserves rhythmic relationships without introducing floating-point rounding errors.

Pitched notes and rests are handled separately. Rest events do not require pitch-related attributes, whereas pitched events must contain a valid pitch step and octave value.

The extracted `MusicEvent` sequence provides the standardized symbolic representation used in the subsequent tokenization and Markov Chain modeling stages.

In [130]:
import xml.etree.ElementTree as ET
from fractions import Fraction


def extract_music_events_from_xml(
    xml_file,
    metadata=None,
):
    """
    Parse a MusicXML document and convert its note and rest
    elements into structured MusicEvent objects.

    Durations are normalized using the MusicXML divisions
    value and represented in quarter-note units.
    """

    metadata = metadata or {}

    tree = ET.parse(xml_file)
    root = tree.getroot()

    events = []
    divisions = 1

    for measure in root.iter():
        measure_tag = measure.tag.split("}")[-1]

        if measure_tag != "measure":
            continue

        measure_number_text = measure.get("number")

        try:
            measure_number = int(measure_number_text)
        except (TypeError, ValueError):
            measure_number = None

        for child in measure:
            child_tag = child.tag.split("}")[-1]

            # Update the active MusicXML divisions value
            if child_tag == "attributes":
                for attribute in child:
                    attribute_tag = attribute.tag.split("}")[-1]

                    if (
                        attribute_tag == "divisions"
                        and attribute.text
                    ):
                        try:
                            parsed_divisions = int(
                                attribute.text.strip()
                            )

                            if parsed_divisions > 0:
                                divisions = parsed_divisions

                        except ValueError:
                            pass

                continue

            if child_tag != "note":
                continue

            is_rest = False
            step = None
            alter = 0.0
            octave = None
            duration_value = None
            lyric = None

            for note_element in child:
                note_tag = note_element.tag.split("}")[-1]

                if note_tag == "rest":
                    is_rest = True

                elif note_tag == "pitch":
                    for pitch_element in note_element:
                        pitch_tag = pitch_element.tag.split("}")[-1]

                        if (
                            pitch_tag == "step"
                            and pitch_element.text
                        ):
                            step = pitch_element.text.strip()

                        elif (
                            pitch_tag == "alter"
                            and pitch_element.text
                        ):
                            try:
                                alter = float(
                                    pitch_element.text.strip()
                                )
                            except ValueError:
                                alter = 0.0

                        elif (
                            pitch_tag == "octave"
                            and pitch_element.text
                        ):
                            try:
                                octave = int(
                                    pitch_element.text.strip()
                                )
                            except ValueError:
                                octave = None

                elif (
                    note_tag == "duration"
                    and note_element.text
                ):
                    try:
                        duration_value = int(
                            note_element.text.strip()
                        )
                    except ValueError:
                        duration_value = None

                elif note_tag == "lyric":
                    for lyric_element in note_element:
                        lyric_tag = lyric_element.tag.split("}")[-1]

                        if (
                            lyric_tag == "text"
                            and lyric_element.text
                        ):
                            lyric = lyric_element.text.strip()
                            break

            if duration_value is None or duration_value <= 0:
                continue

            if not is_rest and (
                step is None or octave is None
            ):
                continue

            normalized_duration = Fraction(
                duration_value,
                divisions,
            )

            event = MusicEvent(
                is_rest=is_rest,
                duration=normalized_duration,
                step=None if is_rest else step,
                alter=None if is_rest else alter,
                octave=None if is_rest else octave,
                lyric=lyric,
                measure_number=measure_number,
                makam=metadata.get("makam"),
                usul=metadata.get("usul"),
                form=metadata.get("form"),
                composer=metadata.get("composer"),
                source_file=xml_file.name,
            )

            events.append(event)

    return events

## Preparing Symbolic Music Sequence Collection

The complete MusicXML corpus is processed to create a structured symbolic music sequence collection.

For each composition:

1. Musical metadata is extracted from the SymbTr filename convention.
2. Symbolic musical events are extracted from the MusicXML representation.
3. Musical events are converted into structured `MusicEvent` sequences.

Each composition is represented as:

- File identifier,
- Musical metadata,
- Sequence of `MusicEvent` objects,
- Symbolic token sequence for probabilistic modeling.

The resulting collection forms the input representation for makam-conditioned statistical analysis and Markov Chain-based music generation.

In [131]:
from tqdm import tqdm

music_sequences = []
failed_xml_files = []

total_files = len(xml_files)

for xml_file in tqdm(
    xml_files,
    desc="Processing MusicXML files",
    unit="file",
):
    try:
        metadata = extract_metadata_from_filename(
            xml_file.name
        )

        events = extract_music_events_from_xml(
            xml_file,
            metadata=metadata,
        )

        if not events:
            raise ValueError(
                "No symbolic events were extracted."
            )

        tokens = events_to_token_sequence(events)

        if not tokens:
            raise ValueError(
                "No symbolic tokens were generated."
            )

        music_sequences.append(
            {
                "file": xml_file.name,
                "metadata": metadata,
                "events": events,
                "tokens": tokens,
                "event_count": len(events),
                "token_count": len(tokens),
            }
        )

    except Exception as error:

        failed_xml_files.append(
            {
                "file": xml_file.name,
                "error": (
                    f"{type(error).__name__}: {error}"
                ),
            }
        )

successful_files = len(music_sequences)
failed_files = len(failed_xml_files)

completion_rate = (
    successful_files / total_files * 100
    if total_files
    else 0.0
)

print("=" * 60)
print("MusicXML Processing Summary")
print("=" * 60)
print(f"Total MusicXML files      : {total_files:,}")
print(f"Successfully processed    : {successful_files:,}")
print(f"Failed files              : {failed_files:,}")
print(f"Success rate              : {completion_rate:.2f}%")
print(f"Failure rate              : {100 - completion_rate:.2f}%")
print("=" * 60)

Processing MusicXML files: 100%|██████████| 3000/3000 [00:21<00:00, 139.51file/s]

MusicXML Processing Summary
Total MusicXML files      : 3,000
Successfully processed    : 3,000
Failed files              : 0
Success rate              : 100.00%
Failure rate              : 0.00%


### Converting MusicEvent Objects into Symbolic Tokens

Each `MusicEvent` object is converted into a compact symbolic token representation.

The token encodes the main musical characteristics required for probabilistic modeling:

- Pitch step,
- Microtonal alteration,
- Octave,
- Duration,
- Rest information.

The token format is:

 D|0.0|5|134.0|0 

represents:

- Step: D
- Alteration: 0.0
- Octave: 5
- Duration: 134
- Rest: False

This representation allows symbolic events to be used as states in Markov Chain models.


In [132]:
if not sample_events:
    raise ValueError(
        "No symbolic events are available for token validation."
    )

sample_event = sample_events[0]
sample_token = sample_event.event_token

print("=" * 60)
print("Sample Symbolic Token Validation")
print("=" * 60)
print(f"Source event : {sample_event}")
print(f"Token        : {sample_token}")
print("=" * 60)
print("events_to_token_sequence loaded successfully.")

Sample Symbolic Token Validation
Source event : MusicEvent(is_rest=False, duration=Fraction(714, 1), step='C', alter=1.59, octave=5, lyric='Al', measure_number=1, makam='acem', usul='duyek', form='ilahi', composer='zekai_dede', source_file='acem--ilahi--duyek--aldanma_dunya--zekai_dede.xml')
Token        : C|1.59000000|5|714/1
events_to_token_sequence loaded successfully.


## Creating Symbolic Music Sequences

Each parsed `MusicEvent` object is transformed into a symbolic token representation to construct the musical sequence used by the Markov Chain model.

The conversion preserves the melodic, rhythmic, and microtonal characteristics encoded in each event while producing a compact representation suitable for statistical sequence modeling.


In [133]:
sample_tokens = events_to_token_sequence(
    sample_events
)

print("=" * 60)
print("Symbolic Token Sequence Validation")
print("=" * 60)
print(f"MusicEvent objects : {len(sample_events):,}")
print(f"Generated tokens   : {len(sample_tokens):,}")
print("=" * 60)

print("First 10 symbolic tokens")
print("-" * 60)

for index, token in enumerate(
    sample_tokens[:10],
    start=1,
):
    print(f"{index:>2}. {token}")

if len(sample_tokens) > 10:
    print(
        f"... ({len(sample_tokens) - 10:,} "
        "additional tokens)"
    )

print("=" * 60)

Symbolic Token Sequence Validation
MusicEvent objects : 262
Generated tokens   : 262
First 10 symbolic tokens
------------------------------------------------------------
 1. C|1.59000000|5|714/1
 2. F|1.70000000|5|536/1
 3. E|1.68000000|5|179/1
 4. G|1.74500000|5|179/1
 5. F|1.70000000|5|179/1
 6. F|1.70000000|5|179/1
 7. E|1.68000000|5|179/1
 8. F|1.70000000|5|714/1
 9. F|1.70000000|5|357/1
10. E|1.68000000|5|179/1
... (252 additional tokens)


## Grouping Compositions by Makam and Usul

The extracted symbolic compositions are grouped according to their musical metadata to enable makam-conditioned Markov Chain modeling.

Each composition is organized based on:

- **Makam:** Modal identity representing the melodic characteristics and pitch organization of the composition.
- **Usul:** Rhythmic cycle structure defining the temporal organization of musical events.

Grouping the corpus by makam and usul enables the models to learn context-specific melodic transition probabilities instead of a single global transition distribution.

The resulting groups provide independent symbolic sequences for training conditional Markov Chain models.

In [134]:
from collections import defaultdict


# Group compositions by makam and usul

makam_groups = defaultdict(list)


for composition in music_sequences:

    metadata = composition["metadata"]


    makam = metadata.get(
        "makam",
        "unknown"
    )


    usul = metadata.get(
        "usul",
        "unknown"
    )


    key = (
        makam,
        usul
    )


    makam_groups[key].append(
        composition
    )



print("=" * 60)

print(
    "Makam Group Statistics"
)

print("=" * 60)


print(
    f"Total makam-usul groups: {len(makam_groups):,}"
)


print(
    f"Total compositions grouped: {sum(len(v) for v in makam_groups.values()):,}"
)


print("=" * 60)


print(
    "Largest makam-usul groups:"
)

print("=" * 60)


for key, compositions in sorted(
    makam_groups.items(),
    key=lambda x: len(x[1]),
    reverse=True
)[:10]:

    print(
        f"{key}: {len(compositions):,} compositions"
    )


print("=" * 60)

Makam Group Statistics
Total makam-usul groups: 1,082
Total compositions grouped: 3,000
Largest makam-usul groups:
('nihavent', 'duyek'): 39 compositions
('huzzam', 'duyek'): 35 compositions
('hicaz', 'duyek'): 34 compositions
('kurdilihicazkar', 'aksak'): 31 compositions
('huzzam', 'aksak'): 29 compositions
('rast', 'duyek'): 27 compositions
('nihavent', 'sofyan'): 26 compositions
('ussak', 'aksak'): 26 compositions
('ussak', 'duyek'): 25 compositions
('hicaz', 'sofyan'): 24 compositions


### Interpretation

The corpus contains **3,000** compositions, which are distributed across **1,082 unique makam–usul combinations**. This indicates a high level of stylistic diversity, as many makam and rhythmic pattern combinations are represented throughout the dataset.

The most common combination is **nihavent–duyek** with **39 compositions**, followed by **huzzam–duyek** (35) and **hicaz–duyek** (34). Among the largest groups, the **Düyek** usul appears most frequently, suggesting that it is the dominant rhythmic pattern in the corpus.

Although some makam–usul combinations contain relatively large numbers of compositions, the majority of combinations are represented by fewer works. Consequently, the dataset exhibits a long-tail distribution, where a small number of combinations are common while many others occur only rarely. This characteristic should be considered when training and evaluating symbolic music generation models.

## Building Makam-Conditioned Markov Chain Models

This section presents two strategies for constructing **makam-conditioned first-order Markov Chain models** for symbolic Turkish makam music generation.

A first-order Markov Chain models the probability of transitioning from one symbolic musical event to the next according to

\[
P(x_{t+1}\mid x_t)
\]

where \(x_t\) denotes the current symbolic state and \(x_{t+1}\) denotes the subsequent state.

Although both strategies employ the same probabilistic framework, they differ in the selection of the training corpus.

### Data Availability-Based Markov Models

The first strategy constructs separate Markov models using the makam categories with the largest number of available compositions. Training on well-represented categories provides a larger number of symbolic sequences, leading to more reliable transition probability estimates and more stable statistical models.

This strategy emphasizes:

- Reliable transition probability estimation,
- Sufficient training data,
- Statistically robust melodic modeling.

### Musical Diversity-Based Markov Models

The second strategy constructs Markov models using makam categories selected to represent a broad range of melodic characteristics. Rather than prioritizing the largest categories, this approach aims to preserve the diversity of the Turkish makam tradition by including modal structures with distinct melodic behaviors.

This strategy emphasizes:

- Representation of diverse makam characteristics,
- Exploration of different melodic organizations,
- Preservation of musical diversity.

The resulting Markov models provide two complementary perspectives for symbolic music generation, enabling a comparison between statistically well-supported models and models designed to capture a wider range of makam characteristics.


### First-Order Markov Chain Model Construction


This section constructs a first-order Markov Chain model from symbolic musical token sequences.
For each current token, the model counts the observed subsequent tokens and converts these transition frequencies into conditional probabilities, defined as
$P(x_{t+1}\mid x_t)=\frac{C(x_t,x_{t+1})}{\sum_j C(x_t,x_j)}$,
where $C(x_t,x_{t+1})$ denotes the number of observed transitions from $x_t$ to $x_{t+1}$.

The resulting model maps each symbolic musical state to a probability distribution over its possible successor states.

In [135]:
from collections import Counter, defaultdict


def build_markov_model(token_sequences):
    """
    Build a first-order Markov Chain model from symbolic
    musical token sequences.

    Parameters
    ----------
    token_sequences : iterable of sequences
        Collection of symbolic token sequences.

    Returns
    -------
    dict
        Mapping from each current token to a probability
        distribution over possible successor tokens.
    """

    transitions = defaultdict(Counter)

    for sequence in token_sequences:
        for current_token, next_token in zip(
            sequence[:-1],
            sequence[1:],
        ):
            transitions[current_token][next_token] += 1

    probabilities = {}

    for current_token, next_counts in transitions.items():
        total_transitions = sum(next_counts.values())

        probabilities[current_token] = {
            next_token: count / total_transitions
            for next_token, count in next_counts.items()
        }

    return probabilities

### Training Makam–Usul-Conditioned Markov Models

The processed compositions are grouped according to their **makam–usul** combinations. For each group, the symbolic token sequences are used to train an independent first-order Markov Chain model.

The resulting models capture the melodic transition characteristics associated with each makam–usul combination.

**Output.** This section reports the total number of makam–usul groups and the number of successfully trained Markov models.

In [136]:
from collections import defaultdict


def group_by_makam_usul(music_sequences):
    """
    Group compositions according to their
    makam–usul combinations.
    """

    groups = defaultdict(list)

    for composition in music_sequences:

        key = (
            composition["metadata"]["makam"],
            composition["metadata"]["usul"],
        )

        groups[key].append(composition)

    return groups


# Group compositions

makam_usul_groups = group_by_makam_usul(
    music_sequences
)


# Train one Markov model for each group

markov_models = {}

for group_key, compositions in makam_usul_groups.items():

    token_sequences = [
        composition["tokens"]
        for composition in compositions
        if composition.get("tokens")
    ]

    if not token_sequences:
        continue

    markov_models[group_key] = build_markov_model(
        token_sequences
    )


print("=" * 60)
print("Markov Model Training Summary")
print("=" * 60)
print(f"Total makam–usul groups     : {len(makam_usul_groups):,}")
print(f"Successfully trained models : {len(markov_models):,}")
print("=" * 60)

Markov Model Training Summary
Total makam–usul groups     : 1,082
Successfully trained models : 1,082


In [137]:
from collections import defaultdict


def group_by_makam_usul(music_sequences):
    """
    Group compositions by their makam–usul combination.
    """

    groups = defaultdict(list)

    for composition in music_sequences:

        key = (
            composition["metadata"]["makam"],
            composition["metadata"]["usul"],
        )

        groups[key].append(composition)


    return groups

## Data Availability-Based Makam–Usul Selection

This strategy selects **makam–usul** groups according to their representation frequency within the corpus.

The number of compositions associated with each **makam–usul** group is calculated and ranked in descending order. Groups with larger numbers of compositions provide more symbolic sequences for estimating transition probabilities, resulting in statistically more reliable Markov Chain models.

**Output.** This cell reports the most highly represented **makam–usul** groups together with the number of available compositions in each group.

In [138]:
# Calculate the number of compositions in each makam–usul group

makam_group_sizes = {

    group_key: len(compositions)

    for group_key, compositions
    in makam_usul_groups.items()

}


# Rank groups by data availability

availability_ranked_groups = sorted(
    makam_group_sizes.items(),
    key=lambda x: x[1],
    reverse=True
)


print("=" * 60)
print("Data Availability-Based Makam–Usul Selection")
print("=" * 60)

print("Top represented makam–usul groups:")

for group, size in availability_ranked_groups[:10]:

    print(
        f"{group[0]} - {group[1]} : {size:,} compositions"
    )

print("=" * 60)

Data Availability-Based Makam–Usul Selection
Top represented makam–usul groups:
nihavent - duyek : 39 compositions
huzzam - duyek : 35 compositions
hicaz - duyek : 34 compositions
kurdilihicazkar - aksak : 31 compositions
huzzam - aksak : 29 compositions
rast - duyek : 27 compositions
nihavent - sofyan : 26 compositions
ussak - aksak : 26 compositions
ussak - duyek : 25 compositions
hicaz - sofyan : 24 compositions


### Interpretation

The ranking reveals that the Turkish makam music corpus is unevenly distributed across **makam–usul** combinations. The most highly represented group is **nihavent–duyek** with **39 compositions**, followed by **huzzam–duyek** (35) and **hicaz–duyek** (34).

Among the ten most represented groups, the **Düyek** usul appears most frequently, indicating that it is the dominant rhythmic cycle in the corpus. In contrast, other usuls, such as **Aksak** and **Sofyan**, are represented by fewer groups and compositions.

These highly represented **makam–usul** groups provide the largest training datasets for first-order Markov Chain modeling, making them suitable candidates for the Data Availability-Based symbolic music generation experiments.

### Selecting Data Availability-Based Markov Models

In [139]:
top_k = 10

selected_availability_groups = [
    group
    for group, size in availability_ranked_groups[:top_k]
]

availability_based_models = {
    group: markov_models[group]
    for group in selected_availability_groups
}

print("=" * 60)
print("Data Availability-Based Model Selection")
print("=" * 60)
print(f"Selected makam–usul groups : {len(selected_availability_groups):,}")
print(f"Selected Markov models     : {len(availability_based_models):,}")
print("=" * 60)

Data Availability-Based Model Selection
Selected makam–usul groups : 10
Selected Markov models     : 10


### Interpretation

A total of **10 makam–usul** groups were selected based on corpus availability. Since the corresponding first-order Markov Chain models had already been trained in the previous section, all **10 pre-trained models** were successfully retrieved for the subsequent symbolic music generation experiments.

This selection strategy ensures that the generation experiments are conducted using models supported by the largest number of training compositions, thereby improving the statistical reliability of the learned transition probabilities while avoiding redundant model training.

## Data Availability-Based Markov Model Statistics

The **Data Availability-Based Markov models** selected in the previous section are summarized by measuring the number of unique symbolic states learned for each **makam–usul** combination.

The number of symbolic states represents the size of the state space associated with each first-order Markov Chain model and provides an indication of its representational capacity. Models with larger state spaces are able to represent a wider range of melodic and rhythmic transitions observed in the training corpus.

Comparing the state-space sizes also provides insight into the diversity of symbolic musical material available for each selected **makam–usul** group.

**Output.** This cell reports the number of unique symbolic states contained in each selected Data Availability-Based Markov model.

In [140]:
print("=" * 60)
print("Data Availability-Based Markov Model Statistics")
print("=" * 60)

availability_model_statistics = []

for group_key, model in availability_based_models.items():

    number_of_states = len(model)

    availability_model_statistics.append(
        {
            "makam": group_key[0],
            "usul": group_key[1],
            "states": number_of_states,
            "model": "Data Availability-Based",
        }
    )

    print(
        f"{group_key[0]} - {group_key[1]} : "
        f"{number_of_states:,} symbolic states"
    )

print("=" * 60)

Data Availability-Based Markov Model Statistics
nihavent - duyek : 855 symbolic states
huzzam - duyek : 757 symbolic states
hicaz - duyek : 822 symbolic states
kurdilihicazkar - aksak : 759 symbolic states
huzzam - aksak : 684 symbolic states
rast - duyek : 568 symbolic states
nihavent - sofyan : 929 symbolic states
ussak - aksak : 504 symbolic states
ussak - duyek : 487 symbolic states
hicaz - sofyan : 535 symbolic states



The **Data Availability-Based Markov models** selected according to corpus availability are summarized in tabular form.

For each selected **makam–usul** group, the table reports:

- **Makam:** Modal category.
- **Usul:** Rhythmic cycle.
- **Symbolic States:** Number of unique symbolic states contained in the selected first-order Markov Chain model.
- **Compositions:** Number of compositions available for the corresponding **makam–usul** group in the training corpus.

This summary provides a quantitative comparison of the selected Markov models together with the amount of training data supporting each model. The table facilitates comparison between model complexity, represented by the number of symbolic states, and corpus availability, represented by the number of training compositions.

In [141]:
model_statistics = []

for group_key, model in availability_based_models.items():

    model_statistics.append(
        {
            "makam": group_key[0],
            "usul": group_key[1],
            "states": len(model),
            "compositions": len(
                makam_usul_groups[group_key]
            ),
        }
    )

model_statistics_df = (
    pd.DataFrame(model_statistics)
    .sort_values(
        by="compositions",
        ascending=False,
    )
    .reset_index(drop=True)
)

# Start row numbering from 1
model_statistics_df.index = (
    model_statistics_df.index + 1
)

model_statistics_df

,makam,usul,states,compositions
1,nihavent,duyek,855,39
2,huzzam,duyek,757,35
3,hicaz,duyek,822,34
4,kurdilihicazkar,aksak,759,31
5,huzzam,aksak,684,29
6,rast,duyek,568,27
7,nihavent,sofyan,929,26
8,ussak,aksak,504,26
9,ussak,duyek,487,25
10,hicaz,sofyan,535,24


## Musical Diversity-Based Markov Models

The second makam-conditioned Markov modeling strategy focuses on representing the musical diversity of Turkish makam music.

Unlike the data availability-based approach, which selects makam-usul groups according to their corpus frequency, this strategy aims to include makam categories with different melodic characteristics.

The objective is to investigate whether Markov Chain models can learn diverse melodic organizations across different makam structures.

The selected makam categories are chosen to represent different modal characteristics, including:

- Common makam families,
- Distinctive melodic behaviors,
- Different pitch organizations,
- Various rhythmic contexts.

This diversity-oriented selection provides a broader evaluation of symbolic music generation beyond frequently represented makam categories.

### Selecting Musically Diverse Makam Categories

The Musical Diversity-Based strategy selects makam-usul groups according to their representative musical characteristics rather than corpus frequency.

The selected categories aim to cover different melodic organizations, modal identities, and rhythmic structures within Turkish makam music.

Unlike the Data Availability-Based approach, this selection does not prioritize the number of available compositions. Instead, it focuses on increasing the representation of diverse makam characteristics in the generation framework.

The selected makam categories provide complementary examples for evaluating whether Markov Chain models can capture different melodic behaviors.

In [142]:
# =====================================================
# Musical Diversity-Based Makam Selection
# =====================================================


# Selected makam-usul groups representing musical diversity


diversity_selected_groups = {


    "rast_duyek":
    (
        "rast",
        "duyek"
    ),


    "hicaz_duyek":
    (
        "hicaz",
        "duyek"
    ),


    "nihavent_duyek":
    (
        "nihavent",
        "duyek"
    ),


    "ussak_duyek":
    (
        "ussak",
        "duyek"
    )

}



print("=" * 60)

print(
    "Musical Diversity-Based Makam Selection"
)

print("=" * 60)



for name, group in diversity_selected_groups.items():

    print(
        f"{name}: {group}"
    )



print("=" * 60)

Musical Diversity-Based Makam Selection
rast_duyek: ('rast', 'duyek')
hicaz_duyek: ('hicaz', 'duyek')
nihavent_duyek: ('nihavent', 'duyek')
ussak_duyek: ('ussak', 'duyek')


### Selecting Musical Diversity-Based Markov Models

The first-order Markov Chain models corresponding to the musically diverse **makam–usul** groups selected in the previous section are retrieved from the complete model collection trained in Section 12.

No additional model training is performed in this step. Instead, the existing models are organized as a diversity-oriented subset for subsequent symbolic music generation experiments.

This strategy enables the comparison of melodic transition behavior across musically distinct makam categories while maintaining a common rhythmic structure.

**Output.** This cell reports the number of musically diverse **makam–usul** groups selected and the number of corresponding Markov models successfully retrieved.

In [143]:
# Store the selected Musical Diversity-Based Markov models

diversity_based_models = {}

missing_diversity_groups = []


for group_name, group_key in diversity_selected_groups.items():

    if group_key not in markov_models:
        missing_diversity_groups.append(group_key)
        continue

    diversity_based_models[group_key] = markov_models[group_key]


print("=" * 60)
print("Musical Diversity-Based Model Selection")
print("=" * 60)
print(
    f"Selected makam–usul groups : "
    f"{len(diversity_selected_groups):,}"
)
print(
    f"Selected Markov models     : "
    f"{len(diversity_based_models):,}"
)
print(
    f"Missing models             : "
    f"{len(missing_diversity_groups):,}"
)
print("=" * 60)

Musical Diversity-Based Model Selection
Selected makam–usul groups : 4
Selected Markov models     : 4
Missing models             : 0


### Inspecting Selected Musical Diversity-Based Groups

The selected **makam–usul** groups are summarized according to the number of compositions available in the training corpus.

This analysis provides an overview of the corpus representation for each musically diverse **makam–usul** category. Unlike the Data Availability-Based strategy, the selected groups are chosen for their musical diversity rather than their frequency in the corpus.

Reporting the number of available compositions helps assess the amount of symbolic data supporting each selected Markov model and facilitates comparison with the Data Availability-Based selection strategy.

**Output.** This cell reports the number of compositions available for each selected Musical Diversity-Based **makam–usul** group.

In [144]:
diversity_group_statistics = []

for group_key in diversity_selected_groups.values():

    if group_key in makam_usul_groups:

        diversity_group_statistics.append(
            {
                "makam": group_key[0],
                "usul": group_key[1],
                "compositions": len(
                    makam_usul_groups[group_key]
                ),
            }
        )

diversity_group_statistics_df = (
    pd.DataFrame(diversity_group_statistics)
    .sort_values(by="compositions", ascending=False)
    .reset_index(drop=True)
)

diversity_group_statistics_df

,makam,usul,compositions
0,nihavent,duyek,39
1,hicaz,duyek,34
2,rast,duyek,27
3,ussak,duyek,25


### Inspecting Musical Diversity-Based Markov Models

The **Musical Diversity-Based Markov models** selected in the previous section are inspected to verify that all selected **makam–usul** groups are represented in the complete collection of trained Markov models.

For each selected **makam–usul** group, the number of unique symbolic states contained in the corresponding first-order Markov Chain model is reported.

This verification step confirms that every musically diverse **makam–usul** category has a valid Markov model available for the subsequent symbolic music generation experiments.

**Output.** This cell reports the number of symbolic states contained in each selected Musical Diversity-Based Markov model.

In [145]:
print("=" * 60)
print("Musical Diversity-Based Markov Model Inspection")
print("=" * 60)

for group_key, model in diversity_based_models.items():

    print(
        f"{group_key[0]} - {group_key[1]} : "
        f"{len(model):,} symbolic states"
    )

print("=" * 60)
print(
    f"Selected Markov models : "
    f"{len(diversity_based_models):,}"
)
print("=" * 60)

Musical Diversity-Based Markov Model Inspection
rast - duyek : 568 symbolic states
hicaz - duyek : 822 symbolic states
nihavent - duyek : 855 symbolic states
ussak - duyek : 487 symbolic states
Selected Markov models : 4


## Generating MIDI Outputs from Selected Markov Models

The **selected first-order Markov Chain models** are used to generate symbolic musical sequences, which are subsequently reconstructed as **MusicEvent** objects and exported as MIDI files.

Two complementary model selection strategies are evaluated:


Generated MIDI files are stored using descriptive filenames that encode the generation strategy together with the corresponding **makam** and **usul**. This naming convention ensures reproducibility and enables every generated composition to be traced back to its source Markov model.

**Output.** The following sections generate MIDI files for each selected strategy and report the number of successfully generated compositions.

In [146]:
import sys
import subprocess

_ = subprocess.run(
    [sys.executable, "-m", "pip", "install", "mido"],
    capture_output=True,
    text=True,
)

In [147]:
from mido import Message, MetaMessage, MidiFile, MidiTrack


STEP_TO_SEMITONE = {
    "C": 0,
    "D": 2,
    "E": 4,
    "F": 5,
    "G": 7,
    "A": 9,
    "B": 11,
}


def music_event_to_midi_note(event):
    """
    Convert a pitched MusicEvent into the nearest
    standard MIDI note number.

    Microtonal alterations are rounded to the nearest
    semitone in this baseline implementation.
    """

    if event.is_rest:
        return None

    base_note = (
        12 * (event.octave + 1)
        + STEP_TO_SEMITONE[event.step]
    )

    midi_note = round(
        base_note + float(event.alter or 0.0)
    )

    return max(
        0,
        min(127, midi_note),
    )


def export_events_to_midi(
    events,
    output_file,
    tempo=500000,
    ticks_per_beat=480,
    duration_scale=1000,
):
    """
    Export MusicEvent objects to a standard MIDI file.

    Parameters
    ----------
    events : iterable
        MusicEvent objects to export.

    output_file : pathlib.Path or str
        Destination MIDI file.

    tempo : int, default=500000
        MIDI tempo in microseconds per quarter note.

    ticks_per_beat : int, default=480
        MIDI temporal resolution.

    duration_scale : int, default=1000
        Scale factor used to convert corpus duration values
        into quarter-note units.
    """

    midi_file = MidiFile(
        ticks_per_beat=ticks_per_beat
    )

    track = MidiTrack()
    midi_file.tracks.append(track)

    track.append(
        MetaMessage(
            "set_tempo",
            tempo=tempo,
            time=0,
        )
    )

    pending_rest_ticks = 0

    for event in events:

        duration_in_quarters = (
            float(event.duration)
            / duration_scale
        )

        duration_ticks = max(
            1,
            round(
                duration_in_quarters
                * ticks_per_beat
            ),
        )

        if event.is_rest:
            pending_rest_ticks += duration_ticks
            continue

        midi_note = music_event_to_midi_note(
            event
        )

        track.append(
            Message(
                "note_on",
                note=midi_note,
                velocity=64,
                time=pending_rest_ticks,
            )
        )

        track.append(
            Message(
                "note_off",
                note=midi_note,
                velocity=64,
                time=duration_ticks,
            )
        )

        pending_rest_ticks = 0

    track.append(
        MetaMessage(
            "end_of_track",
            time=pending_rest_ticks,
        )
    )

    midi_file.save(
        str(output_file)
    )

In [148]:
from pathlib import Path


project_directory = Path.cwd().parent

processed_data_directory = (
    project_directory
    / "data"
    / "processed"
)

generated_midi_directory = (
    processed_data_directory
    / "generated_midi"
)

data_availability_midi_directory = (
    generated_midi_directory
    / "data_availability_based"
)

musical_diversity_midi_directory = (
    generated_midi_directory
    / "musical_diversity_based"
)


data_availability_midi_directory.mkdir(
    parents=True,
    exist_ok=True,
)

musical_diversity_midi_directory.mkdir(
    parents=True,
    exist_ok=True,
)

### Generating MIDI Outputs from Data Availability-Based Markov Models

The **Data Availability-Based Markov models** selected from the most highly represented **makam–usul** groups are used to generate new symbolic musical sequences.

For each selected model, a sequence of symbolic tokens is generated according to the learned first-order transition probabilities:

\[
P(x_{t+1} \mid x_t)
\]

The generated tokens are then converted into **MusicEvent** objects and exported as MIDI files.

The generation pipeline follows these stages:

```text
Selected Data Availability-Based Markov Model
                         ↓
              Symbolic Token Generation
                         ↓
               MusicEvent Reconstruction
                         ↓
                     MIDI Export
```

This strategy evaluates symbolic music generation using Markov models supported by the largest number of compositions in the corpus.

Generated MIDI files are stored in a dedicated output directory. Each filename identifies the generation strategy and the corresponding **makam–usul** group.

**Output.** This cell reports the number of selected models, successfully generated MIDI files, and failed generation attempts.

In [149]:
data_availability_generation_groups = {
    "nihavent_duyek": (
        "nihavent",
        "duyek",
    ),
    "hicaz_duyek": (
        "hicaz",
        "duyek",
    ),
}


generated_data_availability_files = []
failed_data_availability_generations = []


for group_name, group_key in (
    data_availability_generation_groups.items()
):

    try:
        model = availability_based_models[group_key]

        tokens = generate_markov_sequence(
            model,
            length=200,
        )

        events = [
            token_to_music_event(
                token,
                metadata={
                    "makam": group_key[0],
                    "usul": group_key[1],
                },
            )
            for token in tokens
        ]

        midi_file = (
            data_availability_midi_directory
            / f"{group_name}.mid"
        )

        export_events_to_midi(
            events,
            midi_file,
        )

        generated_data_availability_files.append(
            midi_file
        )

    except Exception as error:
        failed_data_availability_generations.append(
            {
                "group": group_key,
                "error": str(error),
            }
        )


print("=" * 60)
print("Data Availability-Based MIDI Generation Summary")
print("=" * 60)
print(
    f"Selected models       : "
    f"{len(data_availability_generation_groups):,}"
)
print(
    f"MIDI files generated  : "
    f"{len(generated_data_availability_files):,}"
)
print(
    f"Failed generations    : "
    f"{len(failed_data_availability_generations):,}"
)
print("=" * 60)

Data Availability-Based MIDI Generation Summary
Selected models       : 2
MIDI files generated  : 2
Failed generations    : 0


### Generating MIDI Outputs from Musical Diversity-Based Markov Models

The **Musical Diversity-Based Markov models** are used to generate new symbolic musical sequences representing different modal characteristics of Turkish makam music.

Unlike the Data Availability-Based strategy, these models are selected according to their musical diversity rather than their frequency in the training corpus. Consequently, the generated compositions represent distinct melodic organizations while maintaining a common rhythmic structure.

For each selected model, symbolic token sequences are generated according to the learned first-order transition probabilities:

\[
P(x_{t+1} \mid x_t)
\]

The generated symbolic tokens are subsequently reconstructed as **MusicEvent** objects and exported as MIDI files.

The generation pipeline follows these stages:

```text
Selected Musical Diversity-Based Markov Model
                          ↓
               Symbolic Token Generation
                          ↓
                MusicEvent Reconstruction
                          ↓
                      MIDI Export
```

Generated MIDI files are stored in a dedicated output directory. Each filename identifies the corresponding **makam–usul** group, ensuring full reproducibility and traceability of the generated compositions.

**Output.** This cell reports the number of selected models, successfully generated MIDI files, and failed generation attempts.

In [150]:
diversity_generation_groups = {
    "rast_duyek": (
        "rast",
        "duyek",
    ),
    "ussak_duyek": (
        "ussak",
        "duyek",
    ),
}


generated_diversity_files = []
failed_diversity_generations = []


for group_name, group_key in (
    diversity_generation_groups.items()
):

    try:
        model = diversity_based_models[group_key]

        tokens = generate_markov_sequence(
            model,
            length=200,
        )

        events = [
            token_to_music_event(
                token,
                metadata={
                    "makam": group_key[0],
                    "usul": group_key[1],
                },
            )
            for token in tokens
        ]

        midi_file = (
            musical_diversity_midi_directory
            / f"{group_name}.mid"
        )

        export_events_to_midi(
            events,
            midi_file,
        )

        generated_diversity_files.append(
            midi_file
        )

    except Exception as error:
        failed_diversity_generations.append(
            {
                "group": group_key,
                "error": str(error),
            }
        )


print("=" * 60)
print("Musical Diversity-Based MIDI Generation Summary")
print("=" * 60)
print(
    f"Selected models       : "
    f"{len(diversity_generation_groups):,}"
)
print(
    f"MIDI files generated  : "
    f"{len(generated_diversity_files):,}"
)
print(
    f"Failed generations    : "
    f"{len(failed_diversity_generations):,}"
)
print("=" * 60)

Musical Diversity-Based MIDI Generation Summary
Selected models       : 2
MIDI files generated  : 2
Failed generations    : 0


In [151]:
print("=" * 60)
print("Generated MIDI Files Summary")
print("=" * 60)
print(
    f"Data Availability-Based : "
    f"{len(generated_data_availability_files):,}"
)
print(
    f"Musical Diversity-Based : "
    f"{len(generated_diversity_files):,}"
)
print(
    f"Total MIDI files         : "
    f"{len(generated_data_availability_files) + len(generated_diversity_files):,}"
)
print("=" * 60)

Generated MIDI Files Summary
Data Availability-Based : 2
Musical Diversity-Based : 2
Total MIDI files         : 4


## Summary of Generated MIDI Files

The symbolic music generation process is completed by exporting the generated musical sequences as MIDI files for both Markov model selection strategies.

The generated MIDI files are summarized according to the following experimental settings:

- **Data Availability-Based Markov Models**, representing the most highly represented **makam–usul** groups in the corpus.
- **Musical Diversity-Based Markov Models**, representing musically distinct **makam–usul** categories.

This summary verifies that the complete symbolic music generation pipeline—from Markov model selection and symbolic sequence generation to MusicEvent reconstruction and MIDI export—has been successfully executed for both strategies.

**Output.** This section reports the number of MIDI files successfully generated for each Markov model selection strategy and the total number of generated MIDI files.

### Listening Instructions

The MIDI files can be played using standard MIDI-compatible software or converted into audio formats for direct listening.

The generated playlist allows comparison between:

- Models trained on highly represented makam categories,
- Models trained on musically diverse makam categories.

The listening evaluation focuses on qualitative differences in:

- Melodic movement,
- Rhythmic organization,
- Makam-specific characteristics,
- Overall musical coherence.

The generated outputs are intended as symbolic AI-generated compositions rather than replacements for traditional makam performance.

## Comparison of Markov Generation Strategies

The two makam-conditioned Markov Chain strategies are compared based on their training characteristics and generated symbolic music properties.

The Data Availability-Based Markov Model prioritizes corpus frequency. Therefore, it benefits from larger training subsets and can estimate transition probabilities from richer symbolic examples.

The Musical Diversity-Based Markov Model prioritizes musical representation. Therefore, it includes makam categories with different melodic identities and provides broader coverage of Turkish makam characteristics.

## Conclusion

This study presents a reproducible computational framework for symbolic analysis and AI-based generation of Turkish makam music.

Two makam-conditioned first-order Markov Chain strategies were investigated:

- Data Availability-Based Markov Models,
- Musical Diversity-Based Markov Models.

The Data Availability-Based approach demonstrated the advantage of learning from larger corpus subsets, providing statistically stable transition models.

The Musical Diversity-Based approach demonstrated the importance of representing different makam characteristics beyond corpus frequency, enabling broader exploration of Turkish makam structures.

The results show that both strategies provide complementary perspectives for computational modeling of cultural music.

Future studies can extend this framework by incorporating higher-order Markov models, deep learning architectures, multimodal representations, and explainable AI approaches for culturally aware music generation.

## Next Chapter

The next chapter, **MIDI Tempo Modification**, adjusts the playback tempo of the generated MIDI files without altering their symbolic musical content.

The tempo-modified MIDI files are then used in the subsequent evaluation workflow.